# Week 1 · Thinking like an RNN

A word-level RNN in pure NumPy — no PyTorch, nothing hidden. We follow the slide's example:

> The **keys** to the old wooden cabinet **are** on the table.

To choose *are* over *is*, the model has to remember *keys* across several words. This notebook shows:

1. **Model** — what the RNN computes at each step: the memory $h_t$ and the prediction $y_t$
2. **Trace** — what it predicts before training
3. **Training** — why we produce $y_t$ at *every* step
4. **Trace again** — what it learned
5. **Generation** — how $y_t$ becomes the next input
6. **Exercise** — testing real long-range memory

Inspired by Andrej Karpathy's `min-char-rnn.py`.

In [1]:
import numpy as np
np.random.seed(0)

## 1 · Data

Eight toy sentences. `<s>` marks the start of a sentence and `</s>` the end, so the model can learn how sentences begin and finish.

In [2]:
corpus = [
    "the keys to the cabinet are on the table",
    "the key to the cabinets is on the table",
    "the keys to the old cabinet are on the table",
    "the key to the old cabinets is on the table",
    "the keys are on the table",
    "the key is on the table",
    "the keys to the cabinet are here",
    "the key to the cabinets is here",
]
sentences = [["<s>"] + s.split() + ["</s>"] for s in corpus]
vocab = sorted({w for s in sentences for w in s})
V = len(vocab)
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}

def one_hot(i):
    x = np.zeros((V, 1)); x[i] = 1
    return x

print(f"{V} words in the vocabulary:", vocab)

14 words in the vocabulary: ['</s>', '<s>', 'are', 'cabinet', 'cabinets', 'here', 'is', 'key', 'keys', 'old', 'on', 'table', 'the', 'to']


## 2 · Model

Two equations, applied at every step $t$:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b_h) \qquad \text{(update the memory)}$$

$$y_t = \mathrm{softmax}(W_y h_t + b_y) \qquad \text{(read the memory out as next-word probabilities)}$$

$y_t$ never feeds into $h$. It's a **readout**: a window onto what the memory currently "thinks" comes next. The same five weight matrices are reused at every step.

In [3]:
H = 16                                   # size of the memory h
Wx = np.random.randn(H, V) * 0.1         # input  -> hidden
Wh = np.random.randn(H, H) * 0.1         # hidden -> hidden (the recurrence)
Wy = np.random.randn(V, H) * 0.1         # hidden -> output (the readout)
bh = np.zeros((H, 1)); by = np.zeros((V, 1))

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

def forward(ids, h0=None):
    h = np.zeros((H, 1)) if h0 is None else h0
    hs, ps = {-1: h}, {}
    for t, i in enumerate(ids):
        hs[t] = np.tanh(Wx @ one_hot(i) + Wh @ hs[t - 1] + bh)   # memory
        ps[t] = softmax(Wy @ hs[t] + by)                         # readout
    return hs, ps

def trace(sentence):
    ids = [w2i[w] for w in ["<s>"] + sentence.split()]
    hs, ps = forward(ids)
    for t, i in enumerate(ids):
        p = ps[t].ravel()
        top = np.argsort(p)[::-1][:3]
        guesses = ", ".join(f"{i2w[k]} ({p[k]:.2f})" for k in top)
        print(f"step {t}: read {i2w[i]!r:11} -> next-word guesses: {guesses}")
    p = ps[len(ids) - 1].ravel()
    print(f"\nP('are') = {p[w2i['are']]:.3f}    P('is') = {p[w2i['is']]:.3f}")

## 3 · Before training

With random weights, every guess is close to $1/V$: the model knows nothing yet.

In [4]:
trace("the keys to the old cabinet")

step 0: read '<s>'       -> next-word guesses: the (0.08), on (0.07), is (0.07)
step 1: read 'the'       -> next-word guesses: here (0.07), </s> (0.07), table (0.07)
step 2: read 'keys'      -> next-word guesses: to (0.08), cabinets (0.08), old (0.07)
step 3: read 'to'        -> next-word guesses: <s> (0.08), </s> (0.07), table (0.07)
step 4: read 'the'       -> next-word guesses: here (0.07), on (0.07), </s> (0.07)
step 5: read 'old'       -> next-word guesses: key (0.07), cabinets (0.07), table (0.07)
step 6: read 'cabinet'   -> next-word guesses: here (0.08), </s> (0.08), on (0.07)

P('are') = 0.067    P('is') = 0.073


## 4 · Training: why $y_t$ at every step

Each sentence becomes many exercises. The input is the sentence without its last word; the target is the same sentence shifted by one:

| step | model has read | $y_t$ should predict |
|---|---|---|
| 1 | `<s>` | the |
| 2 | `<s>` the | keys |
| 3 | `<s>` the keys | to |
| … | … | … |
| 7 | `<s>` the keys to the old cabinet | **are** |

The loss adds up the error of **every** $y_t$. During **backpropagation through time (BPTT)**, the gradient flows from each $y_t$ back through $h_t, h_{t-1}, \dots$ into the shared weights. If $h$ forgot *keys*, the prediction at the last step would be wrong and penalised — that pressure is what teaches the memory what to keep.

In [5]:
lr = 0.1
for epoch in range(301):
    total = 0
    for s in sentences:
        ids = [w2i[w] for w in s]
        inputs, targets = ids[:-1], ids[1:]
        hs, ps = forward(inputs)
        total += -sum(np.log(ps[t][targets[t], 0]) for t in range(len(inputs)))

        # backpropagation through time
        dWx, dWh, dWy = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(Wy)
        dbh, dby = np.zeros_like(bh), np.zeros_like(by)
        dh_next = np.zeros((H, 1))
        for t in reversed(range(len(inputs))):
            dy = ps[t].copy(); dy[targets[t]] -= 1      # error of y_t
            dWy += dy @ hs[t].T; dby += dy
            dh = Wy.T @ dy + dh_next                    # error from y_t + from later steps
            draw = (1 - hs[t] ** 2) * dh                # back through tanh
            dbh += draw; dWx += draw @ one_hot(inputs[t]).T; dWh += draw @ hs[t - 1].T
            dh_next = Wh.T @ draw                       # hand the error to h_{t-1}
        for param, grad in [(Wx, dWx), (Wh, dWh), (Wy, dWy), (bh, dbh), (by, dby)]:
            np.clip(grad, -5, 5, out=grad)
            param -= lr * grad
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  average loss per sentence = {total / len(sentences):.3f}")

epoch   0  average loss per sentence = 21.980


epoch 100  average loss per sentence = 2.816
epoch 200  average loss per sentence = 2.664


epoch 300  average loss per sentence = 2.604


## 5 · After training

Same sentence, trained weights. Watch the guesses become confident, and the verb follow the subject.

In [6]:
trace("the keys to the old cabinet")

step 0: read '<s>'       -> next-word guesses: the (1.00), is (0.00), on (0.00)
step 1: read 'the'       -> next-word guesses: key (0.54), keys (0.45), here (0.00)
step 2: read 'keys'      -> next-word guesses: to (0.74), are (0.26), old (0.00)
step 3: read 'to'        -> next-word guesses: the (1.00), cabinets (0.00), on (0.00)
step 4: read 'the'       -> next-word guesses: cabinet (0.69), old (0.29), are (0.00)
step 5: read 'old'       -> next-word guesses: cabinet (0.98), old (0.01), cabinets (0.00)
step 6: read 'cabinet'   -> next-word guesses: are (1.00), to (0.00), is (0.00)

P('are') = 0.998    P('is') = 0.001


In [7]:
trace("the key to the old cabinets")

step 0: read '<s>'       -> next-word guesses: the (1.00), is (0.00), on (0.00)
step 1: read 'the'       -> next-word guesses: key (0.54), keys (0.45), here (0.00)
step 2: read 'key'       -> next-word guesses: to (0.73), is (0.26), are (0.00)
step 3: read 'to'        -> next-word guesses: the (1.00), table (0.00), on (0.00)
step 4: read 'the'       -> next-word guesses: cabinets (0.69), old (0.30), cabinet (0.00)
step 5: read 'old'       -> next-word guesses: cabinets (0.99), old (0.00), cabinet (0.00)
step 6: read 'cabinets'  -> next-word guesses: is (1.00), to (0.00), are (0.00)

P('are') = 0.001    P('is') = 0.998


## 6 · Generation: $y_t$ becomes the next input

Now $y_t$ is used directly: the most likely word is appended to the text and fed back in as $x_{t+1}$. This loop is how every language model — RNN or Transformer — writes text, one token at a time.

In [8]:
def generate(prompt, max_len=10):
    ids = [w2i[w] for w in ["<s>"] + prompt.split()]
    hs, ps = forward(ids)
    h, p, out = hs[len(ids) - 1], ps[len(ids) - 1], prompt.split()
    for _ in range(max_len):
        nxt = int(np.argmax(p))
        if i2w[nxt] == "</s>":
            break
        out.append(i2w[nxt])
        h = np.tanh(Wx @ one_hot(nxt) + Wh @ h + bh)   # feed the prediction back in
        p = softmax(Wy @ h + by)
    return " ".join(out)

for prompt in ["the keys", "the key to the"]:
    print(f"{prompt!r:18} -> {generate(prompt)}")

'the keys'         -> the keys to the cabinet are here
'the key to the'   -> the key to the cabinets is here


## 7 · Exercise: is it really memory?

**Caveat.** In this corpus, *keys* always appears with *cabinet* and *key* with *cabinets*. The model could get the verb right by looking at the nearest noun instead of remembering the subject.

To test real long-range memory:

1. Add mixed sentences to `corpus`, e.g. `"the keys to the cabinets are on the table"` and `"the key to the cabinet is on the table"`.
2. Rerun all cells and trace `"the keys to the old cabinets"`.
3. Make the gap longer (`"the keys to the very old wooden cabinets"`) and watch when the prediction breaks.

That breaking point is the **vanishing-gradient problem**: the error from the last step shrinks every time it is passed back through $W_h$ and $\tanh$, so early words stop influencing the weights. It's the motivation for LSTMs — and, later, for attention.